## **Read chunks**

In [1]:
import json
path = '/kaggle/input/hotpotqa-distractor'
with open(f'{path}/filtered_chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

## **Semantic-based Retrieval**

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

2025-08-01 09:59:39.446555: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754042379.622161      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754042379.673008      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [3]:
import torch

def semantic_based_retrieval(query, chunks, model, top_k=10):
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    model.to(device)
    
    prefix_query = f"search_query: {query}"
    chunk_texts = [f"search_document: {chunk['chunk_text']}" for chunk in chunks]

    with torch.no_grad():        
        q_vector = model.encode(query, convert_to_tensor=True).unsqueeze(0)
        c_vectors = model.encode(chunk_texts, convert_to_tensor=True)

    q_vector = q_vector.to(device)
    c_vectors = c_vectors.to(device)
    cos = torch.nn.CosineSimilarity(dim=1)
    similarities = cos(q_vector, c_vectors)

    for i, chunk in enumerate(chunks):
        chunk['similarity'] = similarities[i].item()

    return sorted(chunks, key=lambda x: x['similarity'], reverse=True)[:top_k]

query = "In which part of New York City is the director of the romantic comedy 'Big Stone Gap' based?"
semantic_chunks = semantic_based_retrieval(query, chunks, model, 10)
semantic_chunks

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

[{'chunk_id': '5a8e3ea95542995a26add48d_doc9_chunk0',
  'source_doc_title': 'Big Stone Gap (film)',
  'chunk_text': "Big Stone Gap is a 2014 American drama romantic comedy film written and directed by Adriana Trigiani and produced by Donna Gigliotti for Altar Identity Studios, a subsidiary of Media Society.  Based on Trigiani's 2000 best-selling novel of the same name, the story is set in the actual Virginia town of Big Stone Gap circa 1970s.  The film had its world premiere at the Virginia Film Festival on November 6, 2014.",
  'question_id': '5a8e3ea95542995a26add48d',
  'doc_index': 9,
  'chunk_index': 0,
  'similarity': 0.755217432975769},
 {'chunk_id': '5a877e5d5542993e715abf7d_doc8_chunk0',
  'source_doc_title': 'City of Angels (film)',
  'chunk_text': 'City of Angels is a 1998 American romantic fantasy film directed by Brad Silberling and starring Nicolas Cage and Meg Ryan.  Set in Los Angeles, California, the film is a loose remake of Wim Wenders\' 1987 film "Wings of Desire" (

## Save semantic chunks

In [4]:
with open('/kaggle/working/semantic_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(semantic_chunks, f)